# Phase 3.5 — diffusion, as a comparison

**Decides one thing:** GAN or diffusion for the 400 images. Nothing else.

One LoRA on all 2,119 images, ~1 h. Then a Mammalian grid to put beside the
StyleGAN one. If diffusion wins, Phases 1 and 4–7 carry over and Phases 2–3 are
dropped; if it does not, this cost an hour.

| | GPU time |
|---|---|
| GAN path remaining | base 9.2 h + classes 18.4 h ≈ **28 h** |
| This LoRA | **~1 h**, one model, class chosen by prompt |

## What to be suspicious of

Stable Diffusion **already knows what Pokémon look like** — it was trained on
scraped web data. So this LoRA is not learning Pokémon from your 2,119 images;
it is re-surfacing what the base model has, steered by them.

That makes results look better than the data alone justifies, and makes the
Phase 4 memorisation screen *more* necessary, not less. Judge the grid on
**novelty as well as quality** — a beautiful Charizard is a failure here.

## Before running

Export the dataset at 512 (native artwork is 475x475, so this is a 1.08x
upscale rather than the 2x you would get from the 256 set):

```bash
python -m src.data.preprocess --resolution 512
python -m src.data.export_stylegan --resolution 512 --out data/diffusion512 --zip
```

Upload `data/diffusion512.zip` as a Kaggle Dataset and attach it.
**GPU T4 x2**, **Internet On**.

## 1. Settings

In [ ]:
MODEL  = "stable-diffusion-v1-5/stable-diffusion-v1-5"
# If that 404s, try: "CompVis/stable-diffusion-v1-4"
#                    "segmind/small-sd"   (smaller, faster, weaker)

STEPS   = 8000   # ~1 h on one T4. 2,119 images at effective batch 8 ~= 30 epochs.
RANK    = 16     # LoRA rank. 8 is lighter, 32 memorises more.
LR      = 1e-4
RES     = 512

COMPARE   = "mammalian"   # class to generate for the side-by-side
COMPARE_N = 96            # 12 x 8 grid, ~4 min. Raise once it looks promising.

print(f"{STEPS} steps, rank {RANK}, {RES}px")

## 2. Setup

The training script is fetched from the diffusers tag that matches the installed
version, so the two cannot drift apart.

Three of Kaggle's preinstalled packages break this script, all the same way:
something probes whether they are available, and the probe fails rather than
returning `False`.

| package | what breaks |
|---|---|
| `wandb` | broken protobuf stubs; imported whenever merely installed |
| `torchao` | peft's probe **raises** below 0.16.0; Kaggle ships 0.10.0 |
| `gptqmodel` | same raising pattern in peft |

None are used here — this is fp16 LoRA with tensorboard logging — and each probe
returns `False` cleanly once the package is absent, so all three are removed.

**The removals must happen before `import diffusers`**, which is why they live at
the top of this cell. diffusers caches availability at import time, so a kernel
that already imported it keeps the stale answer and later fails with
`ModuleNotFoundError: No module named 'torchao'` from a lazily-loaded submodule.
If you changed packages mid-session, **restart the kernel**.

If the model download 401s, the repo is gated — accept its terms on Hugging Face
and add an `HF_TOKEN` Kaggle secret.

In [ ]:
import os, sys, json, time, pathlib, subprocess
import torch

subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "diffusers[training]", "peft", "accelerate", "transformers",
                "datasets"], check=True)

# Kaggle preinstalls three packages that diffusers/peft probe for and that then
# fail. None are used here -- this is fp16 LoRA with tensorboard logging -- and
# each probe returns False cleanly once the package is ABSENT, so remove them:
#
#   wandb      broken protobuf stubs; the script imports it whenever it is
#              merely installed (line 60), regardless of --report_to
#   torchao    peft's is_torchao_available() RAISES below 0.16.0 instead of
#              returning False, and Kaggle ships 0.10.0
#   gptqmodel  same raising pattern in peft/import_utils.py
for pkg in ["wandb", "torchao", "gptqmodel"]:
    subprocess.run([sys.executable, "-m", "pip", "uninstall", "-q", "-y", pkg],
                   check=False)

import diffusers
VER = diffusers.__version__
SCRIPT = "/kaggle/working/train_text_to_image_lora.py"
url = ("https://raw.githubusercontent.com/huggingface/diffusers/"
       f"v{VER}/examples/text_to_image/train_text_to_image_lora.py")
r = subprocess.run(["wget", "-q", "-O", SCRIPT, url])
if r.returncode != 0 or os.path.getsize(SCRIPT) < 1000:
    url = ("https://raw.githubusercontent.com/huggingface/diffusers/"
           "main/examples/text_to_image/train_text_to_image_lora.py")
    subprocess.run(["wget", "-q", "-O", SCRIPT, url], check=True)
    print("pinned tag not found, used main")

print("diffusers", VER, "| torch", torch.__version__,
      "|", torch.cuda.get_device_name(0))

## 3. Captioned dataset

`train_text_to_image_lora.py` reads an image folder plus a `metadata.jsonl` of
`{"file_name": ..., "text": ...}`.

The caption carries the class, so **one model covers all ten** — you pick the
class at generation time by prompt instead of training ten networks.

In [ ]:
NL = chr(10)
WORD = {"mammalian": "mammal", "arthropod": "insect", "avian": "bird",
        "reptilian": "reptile", "fish": "fish", "amphibian": "amphibian",
        "invertebrate": "invertebrate", "plant_fungus": "plant",
        "mineral_construct": "rock and mineral", "amorphous_ghost": "ghost"}

def caption(cls):
    w = WORD[cls]
    art = "an" if w[0] in "aeiou" else "a"
    return f"{art} {w} pokemon, official artwork, white background"

SRC = next(p.parent for p in pathlib.Path("/kaggle/input").glob("**/summary.json"))
TRAIN = pathlib.Path("/kaggle/working/train")
TRAIN.mkdir(exist_ok=True)

rows = []
for cls in WORD:
    for img in sorted((SRC / cls).glob("*.png")):
        dest = TRAIN / f"{cls}_{img.name}"
        if not dest.exists():
            dest.write_bytes(img.read_bytes())
        rows.append({"file_name": dest.name, "text": caption(cls)})

(TRAIN / "metadata.jsonl").write_text(NL.join(json.dumps(r) for r in rows))
print(f"{len(rows)} captioned images")
print(" ", rows[0]["text"])
print(" ", rows[-1]["text"])

## 4. Train the LoRA

~1 h. One GPU on purpose — same reasoning as the GAN notebook: multi-process
hides tracebacks, and this is a throwaway comparison, not the final run.

`--mixed_precision=fp16` goes to **both** `accelerate` and the script. The
script's own default is `None`, which still resolves to fp16 for the Accelerator
but skips the `if args.mixed_precision == "fp16"` branch that upcasts LoRA
params to fp32 — giving fp16 gradients and a crash at the first gradient clip.

OOM? Drop `--train_batch_size` to 1 and raise `--gradient_accumulation_steps`.

In [ ]:
LORA = "/kaggle/working/lora"
# --mixed_precision must go to BOTH accelerate and the script. The script's own
# default is None, and Accelerator(mixed_precision=None) then falls back to the
# launch flag -- so fp16 is genuinely active -- but the guard that upcasts the
# LoRA params to fp32 is `if args.mixed_precision == "fp16"`, which stays False.
# The result is fp16 gradients and "ValueError: Attempting to unscale FP16
# gradients" at the first clip_grad_norm_.
cmd = ["accelerate", "launch", "--num_processes=1", "--mixed_precision=fp16",
       SCRIPT, "--mixed_precision=fp16",
       f"--pretrained_model_name_or_path={MODEL}",
       f"--train_data_dir={TRAIN}", "--caption_column=text",
       f"--resolution={RES}", "--random_flip",
       "--train_batch_size=2", "--gradient_accumulation_steps=4",
       f"--max_train_steps={STEPS}", f"--rank={RANK}",
       f"--learning_rate={LR}", "--lr_scheduler=constant", "--lr_warmup_steps=0",
       "--seed=0", f"--output_dir={LORA}", "--checkpointing_steps=4000",
       "--dataloader_num_workers=2", "--report_to=tensorboard"]

t0 = time.time()
p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                     text=True, bufsize=1)
for line in p.stdout:
    print(line, end="")
p.wait()
print(f"exit {p.returncode}, {(time.time()-t0)/3600:.2f} h")

## 5. The comparison grid

Generated at 512 and downscaled to 256 — the same size the game will serve, and
the same size as the StyleGAN grid, so the two are directly comparable.

In [ ]:
from diffusers import StableDiffusionPipeline
import PIL.Image
import matplotlib.pyplot as plt

pipe = StableDiffusionPipeline.from_pretrained(
    MODEL, torch_dtype=torch.float16, safety_checker=None).to("cuda")
pipe.set_progress_bar_config(disable=True)
pipe.load_lora_weights(LORA)

prompt = caption(COMPARE)
print(prompt)

imgs, B = [], 8
t0 = time.time()
for i in range(0, COMPARE_N, B):
    n = min(B, COMPARE_N - i)
    g = torch.Generator("cuda").manual_seed(i)
    imgs += pipe([prompt] * n, num_inference_steps=25, guidance_scale=7.5,
                 generator=g).images
print(f"{len(imgs)} images in {(time.time()-t0)/60:.1f} min")

COLS = 12
rows_n = (len(imgs) + COLS - 1) // COLS
sheet = PIL.Image.new("RGB", (COLS * 256, rows_n * 256), "white")
for k, im in enumerate(imgs):
    sheet.paste(im.resize((256, 256), PIL.Image.LANCZOS),
                ((k % COLS) * 256, (k // COLS) * 256))
sheet.save(f"/kaggle/working/diffusion_{COMPARE}.png")

plt.figure(figsize=(18, 18 * sheet.height / sheet.width))
plt.imshow(sheet); plt.axis("off")
plt.title(f"diffusion - {prompt}")
plt.show()

## Verdict

Put this beside `fakes000300.png` from the StyleGAN run and judge three things:

1. **Anatomy.** Do limbs close and faces resolve? This is where the GAN struggled
   at 789 images, and where diffusion should win clearly.
2. **Variety.** Different creatures, or one creature restyled? The GAN showed
   8–12 attractors doing most of the work.
3. **Novelty.** Scan for recognisable real Pokémon. Diffusion is *more* exposed
   here because the base model already knows them — a great-looking Pikachu means
   this path needs a harder memorisation screen, not that it won.

**If diffusion wins:** keep Phase 1 and Phases 4–7, drop Phases 2–3. Class control
becomes a prompt, so the 28 h of per-class training disappears.

**If it does not:** run `train_base()` in `02_finetune.ipynb` and carry on.

Either way, one hour buys an answer to a question that would otherwise hang over
the rest of the project.